# SmartClean Twin - Overrall

**Course:** RBB2013 Digital Twin — May 2026
**Repository:** https://github.com/KAI-UTP/smartclean-twin

**Team Members:**

| No | Name | Student ID |
|---|---|---|
| 1 | Chan Li Kai | 22010900 |
| 2 | William Wong Xiao Kang | 22010943 |
| 3 | Irvin Chang Hou Ceng | 22012342 |
| 4 | Liang Yan Ee | 22011522 |
| 5 | Nurin Emelin Binti Marhisyam | 24006706 |

**Presentation & demo video:** [https://youtu.be/zEq7L-ivMLA](https://youtu.be/zEq7L-ivMLA)

> The video walks through the whole project: problem and purpose, architecture, live Grafana dashboard, NVIDIA Omniverse 3D twin, the five AI models, what-if simulation, live fault injection, command and control, tests, CI, scaling and persistence.

> Prerequisite: docker compose up -d running (8 containers).


## 1. Project at a Glance

Digital Twin of a **mobile cleaning robot** (topic 2 from the project list).

| Layer | Technology | What it does |
|---|---|---|
| Physical asset (simulated) | Python physics simulator | 5m x 5m room, lawnmower path, battery + charging cycle, fault injection |
| Streaming | MQTT (Mosquitto), port 1883 | Telemetry / state / commands, JSON, 1 msg/s |
| Validation | telemetry-ingestion service | Pydantic schema validation before storage |
| Storage | InfluxDB 2.7, port 8086 | Time-series persistence (telemetry, state, predictions, alerts) |
| Twin state | state-engine service | 11-dimension state model (safety, mission, battery, ...) |
| AI | ai-service — 5 models, 3 ML paradigms | Health classification, RUL regression, anomaly detection + forecasts |
| Control | command-api, port 8000 | REST → MQTT commands with acknowledgements |
| Visualization | Grafana (28 panels, 7 sections) + NVIDIA Omniverse 3D | Operator dashboard + live 3D twin |

All services individually containerized (8 containers), with unit / integration /
system / regression test suites and CI on GitHub Actions.


## 1b. Complete Feature Inventory

**Simulated robot (physics, 1s tick):**
- 5m x 5m office room, 10x10 grid, 3 desk obstacles, lawnmower coverage path
- 16 telemetry fields: pose, speed, heading, obstacle distance, battery (V/A/SoC),
  motor current & temperature, dirt score, water level, bumper, brush/pump state
- **Autonomous battery lifecycle**: cleans → SoC < 20% → returns home → charges at dock
  (10%/min) → resumes cleaning at 80% — fully autonomous, no operator input
- **Continuous operation**: after covering the room, the floor "re-dirties" and a new
  cleaning cycle starts — runs forever like a real deployed robot
- Fault injection API (obstacle / motor overload / low battery) for demos & testing

**Data pipeline:**
- MQTT topics with robot-id namespacing (`smartclean/SCR01/...`) — multi-robot ready
- Schema validation (Pydantic) — malformed messages rejected before storage, counted
- InfluxDB persistence — proven to survive container restarts
- Windowed aggregation (30s mean/max, 1m mean) + derivative queries (discharge rate)

**Twin state engine (11 dimensions):**
- motion, operation, safety, battery, cleaning, motor health, dirt level, mission,
  connection, twin quality + coverage % — with alarm generation on rule violations

**Command & control (bidirectional twin):**
- REST → MQTT: START / PAUSE / RESUME / STOP / RETURN_HOME / BRUSH / PUMP
- Every command acknowledged by the robot (accepted/rejected + reason)

**AI layer (5 models + forecasts + advice):** see section 5.

**Visualization:**
- Grafana: 28 panels in 7 collapsible sections (control-room layout)
- NVIDIA Omniverse 3D: live robot pose + heading arrow, coverage tiles turning green,
  battery bar shrinking/recolouring, safety-state body colour, flashing EMERGENCY light,
  red obstacle indicator appearing in front of robot, blue breadcrumb trail

**Engineering:**
- 8 containers, one-command deployment, horizontal scaling of ingestion
- 81 unit + integration + system + regression tests, CI on GitHub Actions
- Full interface contract documented for every service pair


## 2. Architecture

```
 Robot Simulator ──raw──> Mosquitto MQTT <──commands── Command API (REST :8000)
        │                    │      ▲
        │ (1s physics tick)  │      └── acks
        ▼                    ▼
                   Telemetry Ingestion ──validated──> InfluxDB :8086
                             │                           │
                             ▼                           ▼
                        State Engine ──state──>      Grafana :3001
                             │                           ▲
                             ▼                           │
                         AI Service ──predictions────────┘
                     (:8003, 5 models + /whatif)
                                          Omniverse 3D (polls InfluxDB, 1s)
```

Full interface contract (every topic, port, payload schema): `docs/api-contract.md`.


## 3. Live System Check
All 5 application services should answer their health endpoints.

In [1]:
import json, urllib.request

SERVICES = {
    "command-api": "http://localhost:8000/health",
    "telemetry-ingestion": "http://localhost:8001/health",
    "state-engine": "http://localhost:8002/health",
    "ai-service": "http://localhost:8003/health",
    "robot-simulator": "http://localhost:8004/health",
}
for name, url in SERVICES.items():
    try:
        with urllib.request.urlopen(url, timeout=3) as r:
            h = json.loads(r.read())
            print(f"{name:22s} OK   uptime={h.get('uptime_s', '?')}s")
    except Exception as e:
        print(f"{name:22s} DOWN ({e})")


command-api            OK   uptime=227.2s
telemetry-ingestion    OK   uptime=227.2s
state-engine           OK   uptime=227.2s
ai-service             OK   uptime=226.8s
robot-simulator        OK   uptime=?s


## 4. Live Telemetry from InfluxDB
Most recent sensor readings stored by the pipeline (updates every second — re-run the cell).

In [2]:
INFLUX = "http://localhost:8086/api/v2/query?org=smartclean"
TOKEN = "smartclean-super-secret-token"

def flux_query(q):
    req = urllib.request.Request(
        INFLUX, data=q.encode(),
        headers={"Authorization": f"Token {TOKEN}",
                 "Content-Type": "application/vnd.flux",
                 "Accept": "application/csv"})
    with urllib.request.urlopen(req, timeout=5) as r:
        return r.read().decode()

def show_last(measurement):
    q = (f'from(bucket: "smartclean_twin") |> range(start: -30s) '
         f'|> filter(fn: (r) => r._measurement == "{measurement}") |> last()')
    for line in flux_query(q).splitlines():
        p = line.split(",")
        if len(p) > 7 and p[1] == "_result":
            print(f"  {p[7]:28s} = {p[6]}")

print("robot_telemetry:")
show_last("robot_telemetry")


robot_telemetry:
  battery_a                    = 1.5
  battery_soc                  = 99.76
  battery_v                    = 12.594
  brush_on                     = 1
  bumper_active                = 0
  dirt_score                   = 0
  heading_deg                  = 180
  motor_current_a              = 0.9
  motor_temperature_c          = 26.62
  obstacle_cm                  = 200
  pump_on                      = 0
  sequence                     = 206
  speed_mps                    = 0.2
  water_level_pct              = 100
  x_m                          = 3
  y_m                          = 1


### 4b. Live Twin State (11-dimension state model)
The state engine condenses raw telemetry into operator-level states.

In [3]:
print("robot_state:")
show_last("robot_state")


robot_state:
  alarm_count                  = 0
  battery_state                = NORMAL
  cleaning_coverage_pct        = 18.64
  connection_state             = ONLINE
  dirt_level                   = CLEAN
  mission_state                = RUNNING
  motion_state                 = MOVING
  motor_health                 = NORMAL
  safety_state                 = SAFE
  twin_quality                 = SYNCHRONIZED


### 4c. Command & Control — bidirectional twin
The twin doesn't just monitor — it commands the robot. Every command gets an
acknowledgement back from the robot. Demo: PAUSE the robot, verify, RESUME.

In [4]:
import time

def send_command(cmd):
    req = urllib.request.Request(
        "http://localhost:8000/api/v1/commands",
        data=json.dumps({"robot_id": "SCR01", "command": cmd}).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=5) as r:
        resp = json.loads(r.read())
        print(f"  {cmd}: {resp}")

print("Pausing the robot ...")
send_command("PAUSE")
time.sleep(4)
q = ('from(bucket: "smartclean_twin") |> range(start: -5s) '
     '|> filter(fn: (r) => r._measurement == "robot_telemetry" '
     'and r._field == "speed_mps") |> last()')
for line in flux_query(q).splitlines():
    p = line.split(",")
    if len(p) > 7 and p[1] == "_result":
        print(f"  speed_mps while paused = {p[6]}  (expect 0)")
print("Resuming ...")
send_command("RESUME")


Pausing the robot ...
  PAUSE: {'command_id': 'CMD-B8865E20', 'robot_id': 'SCR01', 'command': 'PAUSE', 'status': 'acked', 'request_timestamp': '2026-07-26T13:08:18.838863+00:00', 'ack_received': True, 'ack_timestamp': '2026-07-26T13:08:18.840181+00:00', 'ack_accepted': True, 'ack_reason': None}


  speed_mps while paused = 0  (expect 0)
Resuming ...
  RESUME: {'command_id': 'CMD-60C9B362', 'robot_id': 'SCR01', 'command': 'RESUME', 'status': 'acked', 'request_timestamp': '2026-07-26T13:08:22.873569+00:00', 'ack_received': True, 'ack_timestamp': '2026-07-26T13:08:22.874448+00:00', 'ack_accepted': True, 'ack_reason': None}


## 5. AI Layer — 5 Models, 3 ML Paradigms

| Model | Paradigm | Performance | Predicts |
|---|---|---|---|
| Motor health classifier | Supervised classification | 100% test acc | NORMAL / HIGH_LOAD / OVERHEATED / FAULT |
| Dirt level classifier | Supervised classification | 99.9% | CLEAN / MODERATE / DIRTY |
| Health state classifier | Supervised classification | 90.8% | NORMAL / WARNING / CRITICAL |
| RUL regressor | Supervised regression | R²=0.91, MAE 6.6 min | Remaining useful life (minutes) |
| Anomaly detector | **Unsupervised** | 100% fault detection, 0% false alarms | Never-seen sensor patterns (learned from normal operation only) |

Plus live trend-based forecasts (battery minutes-to-empty, cleaning minutes-to-finish)
and an **operator recommendation** combining all outputs — the twin advises, not just monitors.

Methodology: labelled synthetic datasets with documented physics-motivated rules,
80/20 stratified split, StandardScaler pipelines, cross-checked metrics, models
trained at Docker build time (baked into the image, reproducible).


### 5b. Latest live prediction

In [5]:
print("robot_prediction:")
show_last("robot_prediction")


robot_prediction:
  anomaly_score                = 2.4225
  dirt_level                   = CLEAN
  dirt_level_confidence        = 1
  health_state                 = NORMAL
  health_state_confidence      = 0.9844
  is_anomaly                   = 0
  minutes_to_empty             = 1307.4
  motor_health                 = NORMAL
  motor_health_confidence      = 0.89
  predicted_rul_minutes        = 118.4
  recommendation               = Normal operation — no action needed


## 6. What-If Simulation (core Digital Twin capability)
Test hypothetical scenarios against all 5 models **without touching the robot**.

In [6]:
def whatif(**scenario):
    req = urllib.request.Request(
        "http://localhost:8003/whatif",
        data=json.dumps(scenario).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.loads(r.read())["prediction"]

for label, scenario in [
    ("A: healthy robot", dict(motor_temperature_c=40, motor_current_a=0.8, battery_soc=90)),
    ("B: overheating under load", dict(motor_temperature_c=90, motor_current_a=3.6, battery_soc=40)),
    ("C: low battery + low water", dict(battery_soc=15, water_level_pct=5)),
]:
    p = whatif(**scenario)
    print(f"Scenario {label}")
    print(f"  health={p['health_state_prediction']}  "
          f"RUL={p['predicted_rul_minutes']} min  anomaly={p['is_anomaly']}")
    print(f"  -> {p['recommendation']}")
    print()


Scenario A: healthy robot
  health=NORMAL  RUL=108.8 min  anomaly=False
  -> Normal operation — no action needed



Scenario B: overheating under load
  health=WARNING  RUL=30.3 min  anomaly=True
  -> Sensor anomaly detected — verify sensors and inspect robot



Scenario C: low battery + low water
  health=WARNING  RUL=14.7 min  anomaly=False
  -> Schedule maintenance soon — monitor temperature and load



## 7. Fault Injection Demo (run live)
Injects a motor overload — watch the Grafana overview strip react
(http://localhost:3001/d/smartclean-main), then clears the fault.

In [7]:
import time

def inject(fault):
    req = urllib.request.Request(
        "http://localhost:8004/fault",
        data=json.dumps({"fault": fault}).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=5) as r:
        print(f"POST /fault {fault!r} -> HTTP {r.status}")

inject("motor")
print("Motor overload injected — watch the dashboard overview strip ...")
time.sleep(20)
q = ('from(bucket: "smartclean_twin") |> range(start: -15s) '
     '|> filter(fn: (r) => r._measurement == "robot_prediction" and '
     '(r._field == "recommendation" or r._field == "anomaly_score" '
     'or r._field == "health_state")) |> last()')
for line in flux_query(q).splitlines():
    p = line.split(",")
    if len(p) > 7 and p[1] == "_result":
        print(f"  {p[7]} = {p[6]}")
inject("clear")
print("Fault cleared — dashboard returns to green within ~30 s.")


POST /fault 'motor' -> HTTP 200
Motor overload injected — watch the dashboard overview strip ...


  anomaly_score = -10.8727
  health_state = WARNING
  recommendation = Sensor anomaly detected — verify sensors and inspect robot
POST /fault 'clear' -> HTTP 200
Fault cleared — dashboard returns to green within ~30 s.


## 8. Development Practices Evidence

- **Sprints:** 2 documented cycles — features, milestones, reviews (`docs/sprint-plan.md`)
- **Tests:** 81 unit tests + integration + system + regression suites; pass AND fail cases demonstrated
- **CI/CD:** GitHub Actions — ruff + black lint, full test suite, Docker build on every push
- **Version control:** consistent commit history; teammate review docs pushed from their own accounts
- **Persistence:** proven across container restart (`tests/system/test_persistence.py` — 3/3 pass)
- **Scaling:** `docker compose up --scale telemetry-ingestion=2` (safe — MQTT subscription fan-out)


## 9. Special Features — Beyond the Rubric

These were not required by the project brief:

| # | Feature | Why it's special |
|---|---|---|
| 1 | **NVIDIA Omniverse 3D twin** | Same tech BMW/Siemens use — live USD scene synced to InfluxDB at 1s: robot pose + heading arrow, coverage tiles turning green, shrinking battery bar, safety-state body colour, flashing EMERGENCY light, obstacle indicator, breadcrumb trail |
| 2 | **Autonomous battery lifecycle** | Robot manages its own energy: return home < 20% → dock-charge → resume — no operator needed |
| 3 | **Unsupervised anomaly detection** | 3rd ML paradigm — learned from normal operation only, catches faults it was never shown |
| 4 | **What-if simulation API** | The defining Digital Twin capability — test scenarios without touching the asset (section 6) |
| 5 | **AI recommendation engine** | Twin advises the operator in plain language — monitoring → advising (decision loop closed) |
| 6 | **Operational forecasts** | Live trend-based minutes-to-empty and minutes-to-finish predictions |
| 7 | **Continuous operation loop** | Floor re-dirties after full coverage — twin runs indefinitely like a production system |
| 8 | **Command acknowledgements** | Every command confirmed by the robot — true bidirectional twin, not a dashboard |
